# Week 3 – Vision (Doğukan Yeni): test on new images + automatic plate crops

1. **Week 2 leftover:** test the plate detector on **20 new images** and save good/bad examples.
2. **Week 3:** crop plates automatically with `models/crop_plates.py` and give them to Cemre (Drive `ANPR/auto_crops/`).
3. **If the plate class is weak** (recall was 0.73 on test): try more epochs or `yolov8s`.

**Before you start, put these in Google Drive:**
| Drive path | What |
|---|---|
| `ANPR/models/plate_best.pt` | plate_v1 detector (`best.pt` from the plate run) |
| `ANPR/models/cls_best.pt` | cls_v1 vehicle-type classifier (`best.pt` from the cls run) |
| `ANPR/test_images/` | 20 photos the model has **never seen** (not from train/val/test) |

Runtime → Change runtime type → **T4 GPU**. Run cells one by one.

In [ ]:
!pip install -q ultralytics

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
else:
    !cd anpr-project-en && git pull -q
os.chdir('/content/anpr-project-en')
print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ANPR'
PLATE_WEIGHTS = f'{D}/models/plate_best.pt'
TEST_DIR = f'{D}/test_images'
for p in [PLATE_WEIGHTS, TEST_DIR]:
    print('OK ' if os.path.exists(p) else 'MISSING ', p)

## 1) Test on 20 new images (Week 2 task)
Annotated images are saved to Drive `ANPR/results/week2_20_images/`. Pick 2–3 good and 2–3 bad ones for the slides.

In [ ]:
from ultralytics import YOLO
import glob, matplotlib.pyplot as plt, cv2

model = YOLO(PLATE_WEIGHTS)
print('classes:', model.names)
imgs = sorted(glob.glob(f'{TEST_DIR}/*'))[:20]
res = model.predict(imgs, conf=0.25, save=True, project=f'{D}/results', name='week2_20_images', exist_ok=True, verbose=False)

plt.figure(figsize=(16, 4 * ((len(res) + 3) // 4)))
for i, r in enumerate(res):
    plt.subplot((len(res) + 3) // 4, 4, i + 1)
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f"{os.path.basename(r.path)[:20]}  plates: {len(r.boxes)}", fontsize=8)
plt.tight_layout(); plt.show()

Write down how many images had all plates found, and the bad cases (missed / wrong box). These go into `docs/week4_test_log.md` and the Phase 2 to-do.

## 2) Automatic plate crops for Cemre (Week 3 task)

In [ ]:
!python models/crop_plates.py --weights "$PLATE_WEIGHTS" --source "$TEST_DIR" --out "/content/drive/MyDrive/ANPR/auto_crops" --conf 0.25
import pandas as pd
pd.read_csv(f'{D}/auto_crops/crops.csv').head()

In [ ]:
crops = sorted(glob.glob(f'{D}/auto_crops/*.jpg'))
plt.figure(figsize=(16, 1.6 * ((len(crops) + 5) // 6)))
for i, c in enumerate(crops):
    plt.subplot((len(crops) + 5) // 6, 6, i + 1)
    plt.imshow(cv2.cvtColor(cv2.imread(c), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title(os.path.basename(c)[:22], fontsize=7)
plt.tight_layout(); plt.show()

Tell Cemre the crops are in `ANPR/auto_crops/`. She labels them and measures OCR on real crops (`ocr/week3_ocr_cleanup.ipynb`, step 4).

## 3) Optional: plate class is weak → try more epochs or yolov8s
plate_v1: mAP50 0.82, **recall 0.73** (misses small plates). Run this only if there is time.
Set `DATA_YAML` to the plate dataset's `data.yaml` (Roboflow export unzipped in Drive).

In [ ]:
DATA_YAML = f'{D}/plate_dataset/data.yaml'   # <-- change if your dataset is elsewhere
RUN_THIS = False                               # set True to train (~1 h on T4)

if RUN_THIS:
    m = YOLO('yolov8s.pt')
    m.train(data=DATA_YAML, epochs=80, imgsz=640, project=f'{D}/runs', name='plate_v2_yolov8s')
    met = m.val(data=DATA_YAML, split='test')
    print(f"plate_v2 test: mAP50={met.box.map50:.3f}  mAP50-95={met.box.map:.3f}  P={met.box.mp:.3f}  R={met.box.mr:.3f}")
    print('If better than plate_v1, copy best.pt to ANPR/models/plate_best.pt and add a row to docs/experiments.md')